# 01 — Exploratory Data Analysis

This notebook explores the Superstore Sales dataset to understand its structure,
distributions, and key patterns before building the KPI dashboard.

**Steps covered:**
1. Load & inspect raw data
2. Univariate analysis (distributions)
3. Bivariate analysis (correlations, scatter plots)
4. Time-series overview
5. Key findings summary


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from config import RAW_CSV, CLEAN_CSV

sns.set_theme(style='whitegrid', palette='Blues_r')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 30)

print('Libraries loaded ✅')

## 1. Load & Inspect Raw Data

In [ ]:
df_raw = pd.read_csv(RAW_CSV, encoding='latin-1')
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
print('Missing values:')
df_raw.isnull().sum()[df_raw.isnull().sum() > 0]

In [ ]:
df_raw.describe()

## 2. Load Cleaned Data

In [ ]:
# Run the cleaning pipeline if the cleaned file doesn't exist
if not os.path.exists(CLEAN_CSV):
    from analysis.clean import clean_data
    clean_data()

df = pd.read_csv(CLEAN_CSV, low_memory=False)
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
print(f'Clean shape: {df.shape}')
df.head()

## 3. Revenue Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['sales'], bins=60, ax=axes[0], kde=True, color='steelblue')
axes[0].set_title('Sales Distribution')
axes[0].set_xlabel('Sale Amount ($)')

sns.histplot(df['profit'], bins=60, ax=axes[1], kde=True, color='coral')
axes[1].set_title('Profit Distribution')
axes[1].set_xlabel('Profit ($)')

plt.tight_layout()
plt.show()

## 4. Revenue by Category & Region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_rev = df.groupby('category')['sales'].sum().sort_values()
cat_rev.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Revenue by Category')
axes[0].set_xlabel('Total Revenue ($)')

reg_rev = df.groupby('region')['sales'].sum().sort_values()
reg_rev.plot(kind='barh', ax=axes[1], color='darkcyan')
axes[1].set_title('Revenue by Region')
axes[1].set_xlabel('Total Revenue ($)')

plt.tight_layout()
plt.show()

## 5. Monthly Revenue Trend

In [ ]:
monthly = df.groupby('order_yearmonth')['sales'].sum().reset_index()
monthly.columns = ['month', 'revenue']
monthly = monthly.sort_values('month')

fig = px.line(monthly, x='month', y='revenue',
              title='Monthly Revenue Over Time',
              labels={'revenue': 'Revenue ($)', 'month': 'Month'},
              markers=True)
fig.update_layout(height=400)
fig.show()

## 6. Discount vs Profit Correlation

In [ ]:
fig = px.scatter(df.sample(2000, random_state=42),
                 x='discount', y='profit', color='category',
                 opacity=0.5, trendline='ols',
                 title='Discount vs Profit (sample of 2,000 orders)',
                 labels={'discount': 'Discount Rate', 'profit': 'Profit ($)'})
fig.update_layout(height=420)
fig.show()

## 7. Key Findings

| # | Finding | Supporting Data |
|---|---|---|
| 1 | **West region** generates the highest revenue but Central has the highest profit margin | `01_revenue_by_region.sql` |
| 2 | **Technology** is the most profitable category; **Furniture** the least | `04_category_breakdown.sql` |
| 3 | High discounts (≥40%) consistently produce **negative profit** | `08_anomaly_detection.sql` |
| 4 | **Consumer** segment accounts for ~50% of all orders | `05_customer_segments.sql` |
| 5 | Revenue has a seasonal peak in **Q4** each year | Monthly trend chart |
